In [ ]:
# YouTube Video Notes Generator

## 1 — Project Overview

This notebook builds a **YouTube transcript-to-notes pipeline** that takes any YouTube video URL and produces:
- **Structured chaptered notes** with timestamps
- **Key takeaways** summary
- **Action items** for the viewer
- **Quiz questions** for self-testing
- **Glossary** of technical terms

**The engineering pattern:** This is a classic **map-reduce summarization** pipeline — one of the most common GenAI patterns in production. We split a long document into sections (map), summarize each independently, then combine results (reduce).

**Stack:**
| Component | Tool | Why |
|-----------|------|-----|
| Transcript fetch | youtube-transcript-api | Free, no API key, reliable |
| Orchestration | LangChain (LCEL) | Clean chain composition |
| LLM | Ollama (local) | Free, offline, no API keys |
| Output | Markdown | Universal, readable, portable |

**Prerequisite:** [Ollama](https://ollama.com/) installed and running with a model pulled (e.g., `ollama pull qwen3:8b`).

## 2 — Learning Goals

By the end of this notebook you will be able to:

1. **Fetch YouTube transcripts** programmatically — no API key needed
2. **Design effective prompts** — understand prompt structure, instructions, format control, and few-shot examples
3. **Build a map-reduce summarization pipeline** — the standard pattern for summarizing documents longer than an LLM's context window
4. **Use LangChain Expression Language (LCEL)** — compose prompt → LLM → parser chains cleanly
5. **Generate multiple output formats from the same input** — notes, quiz questions, action items — by changing only the prompt
6. **Compare naive (single-prompt) vs. structured (multi-step) approaches** — understand why chunked pipelines produce better results
7. **Identify common prompt engineering mistakes** — vague instructions, missing format specs, no grounding

## 3 — Problem Statement

**Problem:** You watch a 30-minute technical video. Two days later, you remember only the vibe. You want structured, searchable notes — but manual note-taking is slow and you miss details.

**Constraints:**
- YouTube transcripts are noisy (auto-generated, no punctuation, filler words)
- Videos can be 10 minutes or 3 hours — the approach must scale
- Notes should be structured, not wall-of-text summaries
- Different use cases need different formats (study notes vs. action items vs. quiz)

**Our approach:**
```
YouTube URL → Fetch transcript → Clean/group into time sections
    → Summarize each section (map) → Combine into final notes (reduce)
    → Optionally generate quiz questions / action items with separate prompts
```

In [ ]:
## 4 — Why This Project Matters

**Map-reduce summarization is everywhere in production GenAI:**
- **Notion AI / Google Docs AI** — summarize long documents by chunking
- **Meeting recorders** (Otter.ai, Fireflies) — same pattern on meeting transcripts
- **Legal tech** — summarize depositions, contracts, case files
- **Podcast tools** — generate show notes and episode summaries

**Why not just paste the whole transcript into ChatGPT?**
- Transcripts can exceed context windows (a 1-hour video ≈ 10K words ≈ 13K tokens)
- Even if it fits, LLMs degrade on very long inputs ("lost in the middle" phenomenon)
- Section-by-section summaries preserve timeline structure and timestamps
- Separate prompts for different outputs (notes vs quiz) give better results than one mega-prompt

**Prompt engineering is the core skill here.** The difference between a useful output and a useless one is almost always the prompt, not the model.

## 5 — Data Overview

**Our "dataset" is a YouTube video transcript.** We'll use a well-known public video for demonstration.

**What `youtube-transcript-api` returns:**
Each transcript is a list of segments, where each segment has:
- `text` — the spoken words (may be auto-generated, so expect noise)
- `start` — start time in seconds
- `duration` — segment length in seconds

**Characteristics of auto-generated transcripts:**
- No punctuation or capitalization (often)
- Filler words included ("um", "uh", "like")
- Occasional errors in technical terms
- Segments are very short (2-5 seconds each, ~5-15 words)

We need to group these tiny segments into meaningful sections for summarization.

In [ ]:
## 6 — Environment Setup

**Required services:**
- **Ollama** must be installed and running locally: https://ollama.com/
- Pull a model: `ollama pull qwen3:8b` (or any model you prefer)

**No API keys needed** — both `youtube-transcript-api` and Ollama are free and local.

# Install dependencies (uncomment if needed)
# !pip install youtube-transcript-api langchain langchain-ollama -q

import subprocess, shutil

# Verify Ollama is available
if shutil.which("ollama") is None:
    print("WARNING: Ollama not found in PATH. Install from https://ollama.com/")
    print("After installing, run: ollama pull qwen3:8b")
else:
    print("Ollama found.")

print("Setup complete.")

In [ ]:
## 7 — Imports

import re
import textwrap
from pathlib import Path

from youtube_transcript_api import YouTubeTranscriptApi
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("All imports successful!")

In [ ]:
## 8 — Data Loading (Fetch Transcript)

We extract the video ID from the URL and fetch the transcript using `youtube-transcript-api`. This library scrapes YouTube's subtitle endpoint directly — no Google API key required.

**Important:** This only works for videos that have subtitles (most popular videos have auto-generated captions).

def extract_video_id(url: str) -> str:
    """Extract the 11-character video ID from various YouTube URL formats."""
    patterns = [
        r'(?:v=|/v/|youtu\.be/)([\w-]{11})',   # standard + short URLs
        r'(?:embed/)([\w-]{11})',                # embed URLs
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Could not extract video ID from: {url}")


def format_timestamp(seconds: float) -> str:
    """Convert seconds to MM:SS or HH:MM:SS format."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    if hours > 0:
        return f"{hours}:{minutes:02d}:{secs:02d}"
    return f"{minutes}:{secs:02d}"


def get_transcript(video_url: str) -> list[dict]:
    """Fetch transcript for a YouTube video. Prefers English captions."""
    video_id = extract_video_id(video_url)
    print(f"Video ID: {video_id}")
    transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])
    return transcript


# ── Demo video ──
# 3Blue1Brown's "But what is a neural network?" — well-known, public, has captions
VIDEO_URL = "https://www.youtube.com/watch?v=aircAruvnKk"

transcript = get_transcript(VIDEO_URL)
total_duration = transcript[-1]["start"] + transcript[-1]["duration"]

print(f"\nFetched {len(transcript)} segments")
print(f"Total duration: {format_timestamp(total_duration)}")
print(f"\nFirst 5 segments:")
for seg in transcript[:5]:
    ts = format_timestamp(seg["start"])
    print(f"  [{ts}] {seg['text']}")

In [ ]:
## 9 — Data Inspection

Let's understand the structure and quality of the raw transcript before processing.

# Inspect transcript quality
full_text = " ".join(seg["text"] for seg in transcript)
word_count = len(full_text.split())
avg_seg_words = word_count / len(transcript)
avg_seg_duration = sum(seg["duration"] for seg in transcript) / len(transcript)

print(f"Total word count: {word_count:,}")
print(f"Average words per segment: {avg_seg_words:.1f}")
print(f"Average segment duration: {avg_seg_duration:.1f}s")
print(f"Estimated tokens (words * 1.3): ~{int(word_count * 1.3):,}")
print(f"\n--- Raw text preview (first 500 chars) ---")
print(full_text[:500])
print(f"\n--- Middle of transcript (chars 2000-2500) ---")
print(full_text[2000:2500])

In [ ]:
# Combine section summaries
combined_summaries = "\n\n".join(
    f"[{s['timestamp']}]\n{s['summary']}" for s in section_summaries
)

# Final notes prompt
final_prompt = ChatPromptTemplate.from_template(
    """You are a study notes assistant. Given the section-by-section summaries of a video,
create comprehensive, well-organized notes.

Your output should include:

1. **OVERVIEW** — A 2-3 sentence summary of the entire video
2. **CHAPTERS** — Keep the chaptered summaries with timestamps
3. **KEY TAKEAWAYS** — The 5 most important points from the entire video
4. **ACTION ITEMS** — Things the viewer should do, try, or explore further
5. **GLOSSARY** — Define any technical terms mentioned

Section summaries:
{summaries}
"""
)

final_chain = final_prompt | llm | StrOutputParser()

print("Generating final notes...")
final_notes = final_chain.invoke({"summaries": combined_summaries})
print("Done!\n")
print("=" * 70)
print(final_notes)
print("=" * 70)

## Step 7 — Save Notes to File

In [ ]:
from pathlib import Path

output_path = Path("video_notes.md")
output_path.write_text(
    f"# Video Notes\n\n"
    f"**Source:** {VIDEO_URL}\n\n"
    f"---\n\n"
    f"{final_notes}",
    encoding="utf-8",
)
print(f"Notes saved to: {output_path.absolute()}")

## Step 8 — Interactive Mode (Try Your Own Videos!)

In [ ]:
def generate_notes(video_url: str) -> str:
    """Complete pipeline: URL → structured notes."""
    print(f"🎬 Processing: {video_url}")
    
    # 1. Fetch transcript
    transcript = get_transcript(video_url)
    print(f"  📝 Got {len(transcript)} transcript segments")
    
    # 2. Group into sections
    sections = group_transcript_into_sections(transcript)
    print(f"  📂 Grouped into {len(sections)} sections")
    
    # 3. Summarize each section
    summaries = []
    for i, section in enumerate(sections):
        ts = format_timestamp(section["start"])
        summary = section_chain.invoke({"timestamp": ts, "text": section["text"]})
        summaries.append({"timestamp": ts, "summary": summary})
        print(f"  ✓ Section {i+1}/{len(sections)}")
    
    # 4. Generate final notes
    combined = "\n\n".join(f"[{s['timestamp']}]\n{s['summary']}" for s in summaries)
    notes = final_chain.invoke({"summaries": combined})
    
    print(f"  ✅ Notes generated!")
    return notes


# Uncomment and try with your own video:
# my_notes = generate_notes("https://www.youtube.com/watch?v=YOUR_VIDEO_ID")
# print(my_notes)

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **youtube-transcript-api** | Fetches subtitles/captions from YouTube without an API key |
| **Section Grouping** | Combines tiny transcript segments into meaningful time-based sections |
| **ChatPromptTemplate** | Defines structured prompts with input variables |
| **StrOutputParser** | Extracts plain text from LLM response objects |
| **Chain (prompt \| llm \| parser)** | LangChain Expression Language (LCEL) for composing steps |
| **Map-Reduce Pattern** | Summarize each section individually, then combine all summaries |

## 🔧 Customization Ideas

- **Change section duration**: `section_duration=600` for 10-minute chapters
- **Different output format**: Modify the final prompt for flashcards, quiz questions, etc.
- **Multi-language**: Change `languages=["en"]` to support other languages
- **Batch processing**: Loop over a playlist of video URLs